# LangChain Chat Model Reference

Developer-facing statements defined in `langchain_core.language_models.chat_models`.

# `generate_from_stream`

Combines synchronous `ChatGenerationChunk` values into one `ChatResult`.

## Syntax

```python
generate_from_stream(
    stream: Iterator[ChatGenerationChunk], # Chat generation chunks to combine
) -> ChatResult # Return the combined chat result
```

Raises `ValueError` when the stream contains no chunks.

---

# `agenerate_from_stream`

Asynchronously combines `ChatGenerationChunk` values into one `ChatResult`.

## Syntax

```python
async agenerate_from_stream(
    stream: AsyncIterator[ChatGenerationChunk], # Asynchronous chat generation chunks
) -> ChatResult # Return the combined chat result
```

Raises `ValueError` when the stream contains no chunks.

# `BaseChatModel: BaseLanguageModel[AIMessage], ABC`

`BaseChatModel` is the abstract base class for conversational language models.

Concrete subclasses must implement `_generate()` and `_llm_type`.

## Fields

```python
rate_limiter: BaseRateLimiter | None = None # Optional request rate limiter
disable_streaming: bool | Literal["tool_calling"] = False # Control when streaming is bypassed
output_version: str | None = None # AIMessage content format; commonly "v0" or "v1"
profile: ModelProfile | None = None # Optional model capability profile
```

## Overridden Properties and Methods

### `OutputType`

Returns `AnyMessage` as the Runnable output type.

### `invoke`

Synchronously accepts a string, message sequence, or `PromptValue` and returns one `AIMessage`.

### `ainvoke`

Asynchronously accepts a string, message sequence, or `PromptValue` and returns one `AIMessage`.

### `stream`

Synchronously yields `AIMessageChunk` values.

Falls back to `invoke()` when streaming is unavailable or disabled.

### `astream`

Asynchronously yields `AIMessageChunk` values.

Falls back to `ainvoke()` when streaming is unavailable or disabled.

### `stream_events`

Returns normal `StreamEvent` values for versions `"v1"` and `"v2"`.

Returns a beta `ChatModelStream` for version `"v3"`.

### `astream_events`

Returns normal asynchronous `StreamEvent` values for versions `"v1"` and `"v2"`.

Returns an awaitable `AsyncChatModelStream` for version `"v3"`.

### `generate_prompt`

Converts each `PromptValue` into messages and delegates to `generate()`.

### `agenerate_prompt`

Asynchronously converts each `PromptValue` into messages and delegates to `agenerate()`.

### `bind`

Binds keyword arguments while preserving chat-model-specific event-stream typing.

## Methods

### `generate`

Generates results for multiple lists of chat messages.

```python
generate(
    messages: list[list[BaseMessage]], # Batched chat-message inputs
    stop: list[str] | None = None, # Optional stop sequences
    callbacks: Callbacks = None, # Callback handlers
    *,
    tags: list[str] | None = None, # Run tags
    metadata: dict[str, Any] | None = None, # Run metadata
    run_name: str | None = None, # Optional run name
    run_id: UUID | None = None, # Optional run identifier
    **kwargs: Any, # Provider-specific generation arguments
) -> LLMResult # Return batched generations and model output
```

### `agenerate`

Asynchronously generates results for multiple lists of chat messages.

```python
async agenerate(
    messages: list[list[BaseMessage]], # Batched chat-message inputs
    stop: list[str] | None = None, # Optional stop sequences
    callbacks: Callbacks = None, # Callback handlers
    *,
    tags: list[str] | None = None, # Run tags
    metadata: dict[str, Any] | None = None, # Run metadata
    run_name: str | None = None, # Optional run name
    run_id: UUID | None = None, # Optional run identifier
    **kwargs: Any, # Provider-specific generation arguments
) -> LLMResult # Return batched generations and model output
```

### `dict`

Deprecated since `1.4.2` and scheduled for removal in `2.0.0`.

Use `asdict()` instead.

### `asdict`

Returns the identifying model parameters together with `_type`.

```python
asdict(
    self, # Current chat model
) -> dict[str, Any] # Return the model dictionary
```

### `bind_tools`

Binds tool definitions to models that support tool calling.

```python
bind_tools(
    tools: Sequence[
        dict[str, Any] | type | Callable[..., Any] | BaseTool
    ], # Tools or tool schemas to bind
    *,
    tool_choice: str | None = None, # Optional tool-selection rule
    **kwargs: Any, # Model-specific binding arguments
) -> Runnable[LanguageModelInput, AIMessage] # Return the tool-enabled model
```

The base implementation raises `NotImplementedError`.

### `with_structured_output`

Creates a Runnable that parses model output using a Pydantic, `TypedDict`, JSON, or OpenAI tool schema.

```python
with_structured_output(
    schema: dict[str, Any] | type, # Required output schema
    *,
    include_raw: bool = False, # Whether raw output and parsing errors are included
    **kwargs: Any, # Compatibility arguments
) -> Runnable[
    LanguageModelInput,
    dict[str, Any] | BaseModel,
] # Return the structured-output Runnable
```

When `include_raw=True`, output contains `raw`, `parsed`, and `parsing_error`.

Raises `NotImplementedError` when the model does not implement tool binding.

## Required Subclass Hooks

### `_generate`

Generates one `ChatResult` from one list of messages.

```python
_generate(
    self, # Current chat model
    messages: list[BaseMessage], # Input messages
    stop: list[str] | None = None, # Optional stop sequences
    run_manager: CallbackManagerForLLMRun | None = None, # Synchronous callback manager
    **kwargs: Any, # Provider-specific generation arguments
) -> ChatResult # Return the generated chat result
```

### `_llm_type`

Returns the string used to identify the chat-model type.

```python
@property
def _llm_type(
    self, # Current chat model
) -> str # Return the model type identifier
```

## Optional Subclass Hooks

### `_resolve_model_profile`

Returns the automatically resolved model capability profile.

### `_agenerate`

Provides native asynchronous generation.

The default implementation runs `_generate()` in an executor.

### `_stream`

Provides synchronous generation streaming.

The default implementation raises `NotImplementedError`.

### `_astream`

Provides asynchronous generation streaming.

The default implementation adapts `_stream()` through an executor.

## Behaviour

- Accepts strings, `PromptValue` objects, and message sequences.
- Supports synchronous, asynchronous, batch, and streaming execution.
- Applies callbacks, tags, metadata, caching, and rate limiting.
- `disable_streaming=True` always bypasses streaming.
- `disable_streaming="tool_calling"` bypasses streaming when tools are supplied.
- `output_version="v1"` stores standardized content blocks in message content.
- Version `"v3"` event streaming is beta and returns typed chat-model stream objects.


In [ ]:
from collections.abc import Iterator # Import Iterator for synchronous streaming
from typing import Any # Import Any for provider-specific arguments

from langchain_core.callbacks import CallbackManagerForLLMRun # Import the callback manager
from langchain_core.language_models.chat_models import BaseChatModel # Import the abstract chat-model base class
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage # Import chat message types
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult # Import chat-model result types


class UppercaseChatModel(BaseChatModel): # Create a concrete BaseChatModel implementation
    prefix: str = "Assistant: " # Store a configurable response prefix

    @property # Define the model-type identifier as a property
    def _llm_type(self) -> str: # Implement the required model-type property
        return "uppercase-chat-model" # Return the custom model identifier

    @property # Define parameters used for tracing and serialization
    def _identifying_params(self) -> dict[str, Any]: # Return this model's identifying configuration
        return {"prefix": self.prefix} # Include the configured response prefix

    def _create_response(self, messages: list[BaseMessage]) -> str: # Build a response from chat messages
        latest_text: str = str(messages[-1].content) # Read the latest message content
        return f"{self.prefix}{latest_text.upper()}" # Return the uppercase response

    def _generate( # Implement the required synchronous generation method
        self, # Current chat-model instance
        messages: list[BaseMessage], # Messages supplied to the model
        stop: list[str] | None = None, # Optional stop sequences
        run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
        **kwargs: Any, # Additional model arguments
    ) -> ChatResult: # Return one complete chat result
        response: str = self._create_response(messages) # Generate the response text

        if stop: # Check whether stop sequences were supplied
            for stop_sequence in stop: # Process every stop sequence
                response = response.split(stop_sequence)[0] # Remove text after the stop sequence

        message: AIMessage = AIMessage(content=response) # Create the final assistant message
        generation: ChatGeneration = ChatGeneration(message=message) # Wrap the message as a generation
        return ChatResult(generations=[generation]) # Return the completed chat result

    def _stream( # Implement optional synchronous streaming
        self, # Current chat-model instance
        messages: list[BaseMessage], # Messages supplied to the model
        stop: list[str] | None = None, # Optional stop sequences
        run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
        **kwargs: Any, # Additional model arguments
    ) -> Iterator[ChatGenerationChunk]: # Yield chat-generation chunks
        response: str = self._create_response(messages) # Generate the complete response text

        if stop: # Check whether stop sequences were supplied
            for stop_sequence in stop: # Process every stop sequence
                response = response.split(stop_sequence)[0] # Remove text after the stop sequence

        for word in response.split(): # Process every response word
            chunk: AIMessageChunk = AIMessageChunk(content=f"{word} ") # Create one message chunk
            yield ChatGenerationChunk(message=chunk) # Yield the wrapped generation chunk

        return # Finish the generator


model: UppercaseChatModel = UppercaseChatModel( # Create the custom chat model
    prefix="Model reply: ", # Configure its response prefix
) # Finish creating the model

invoke_result: AIMessage = model.invoke("hello langchain") # Invoke the model with a string

batch_results: list[AIMessage] = model.batch( # Invoke the model with multiple inputs
    [
        "python is useful", # Supply the first input
        "langchain supports runnables", # Supply the second input
    ]
) # Finish the batch invocation

print("Invoke result:", invoke_result.content) # Display the normal invocation result

for result in batch_results: # Iterate through the batch responses
    print("Batch result:", result.content) # Display each batch response

print("Stream result:", end=" ") # Display the streaming heading

for chunk in model.stream("stream this response"): # Stream message chunks from the model
    print(chunk.content, end="") # Display each generated chunk immediately

print() # Move to the next output line

print("Model details:", model.asdict()) # Display the model's identifying parameters

# `SimpleChatModel: BaseChatModel`

`SimpleChatModel` is an abstract compatibility base class for chat models that return plain text.

New model implementations should generally inherit directly from `BaseChatModel`.

## Required Subclass Method

### `_call`

Generates one text response from a list of messages.

```python
_call(
    self, # Current chat model
    messages: list[BaseMessage], # Input messages
    stop: list[str] | None = None, # Optional stop sequences
    run_manager: CallbackManagerForLLMRun | None = None, # Synchronous callback manager
    **kwargs: Any, # Provider-specific generation arguments
) -> str # Return the generated text
```

## Implemented Methods

### `_generate`

Calls `_call()` and wraps the returned text in an `AIMessage`, `ChatGeneration`, and `ChatResult`.

### `_agenerate`

Runs `_generate()` asynchronously in an executor.

A concrete subclass must also implement the inherited `_llm_type` property.

In [ ]:
from typing import Any # Import Any for additional model arguments

from langchain_core.callbacks import CallbackManagerForLLMRun # Import the callback manager
from langchain_core.language_models.chat_models import SimpleChatModel # Import the abstract SimpleChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage # Import LangChain message types


class ReverseChatModel(SimpleChatModel): # Create a concrete SimpleChatModel implementation
    prefix: str = "Model: " # Store a prefix added to every response

    @property # Define the inherited abstract property
    def _llm_type(self) -> str: # Return the custom model type
        return "reverse-chat-model" # Return the model identifier

    def _call( # Implement the required text-generation method
        self, # Current model instance
        messages: list[BaseMessage], # Conversation messages received by the model
        stop: list[str] | None = None, # Optional stop sequences
        run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
        **kwargs: Any, # Additional generation arguments
    ) -> str: # Return the generated text
        latest_message: str = str(messages[-1].content) # Read the latest message content

        response: str = f"{self.prefix}{latest_message[::-1]}" # Reverse the latest message

        if stop: # Check whether stop sequences were supplied
            for stop_sequence in stop: # Process each stop sequence
                response = response.split(stop_sequence)[0] # Remove text after the stop sequence

        return response # Return the final response text


model: ReverseChatModel = ReverseChatModel( # Create the custom chat model
    prefix="Assistant: ", # Configure the response prefix
) # Finish creating the model

string_result: AIMessage = model.invoke("Hello LangChain") # Invoke the model using a string

message_result: AIMessage = model.invoke( # Invoke the model using a message list
    [
        HumanMessage(content="Python"), # Supply a human message
    ]
) # Finish invoking the model

batch_results: list[AIMessage] = model.batch( # Process multiple inputs
    [
        "First message", # Supply the first input
        "Second message", # Supply the second input
    ]
) # Finish the batch execution

print("String result:", string_result.content) # Display the string-input result

print("Message result:", message_result.content) # Display the message-input result

for result in batch_results: # Iterate through the batch results
    print("Batch result:", result.content) # Display each generated response

print("Model type:", model._llm_type) # Display the custom model identifier